So the issue with using a random terr to determine time until next res tick is that you can only deduce that the res tick is within, lets say, [n,n+4] where n is divisible by 4. Which is why we need to use something that produces res every once per second rather than once per 4s while having complete information on the res usage, and what's better to do that than cities?

The idea: Given a city with 3/3 em prod (ems >= 288k) and no resource prod (<=4.5k res prod) (treasury bonus requires calculation to distance to hq which can be annoying), subtract the ems spent on storages to obtain the net ems gain per hour. Let g be the generation per hour and d be the usage per hour, at the start of each res tick a terr gians d/60 from hq and generates g/3600 per second while spending d/3600 per second. Meaning, time after last res tick (t) in seconds is calculated as r = d/60-dt/3600+gt/3600 where r is the current res in the hq

hence:

r - d/60 = gt/3600 - dt/3600

r - d/60 = t (g/3600 - d/3600)

t = (r-d/60) / (g/3600 - d/3600)

lvl2 storage = 800 ems / hr -> 1200 max on each res

lvl3 storage = 3000 ems / hr -> 2400 max on each res

In [ ]:
import requests
import statistics
import numpy as np

r = requests.get('https://api.wynncraft.com/v3/guild/list/territory')
data = r.json()
list_of_cities = ['Ragni','Nemract','Detlas','Troms','Nesaak','Lusuco','Lutho','Llevigar','Olux','Gelibord','Cinfras','Thesead','Thanos','Kandon-Beda','Rodoroc','Ahmsord','Selchar','Corkus City','Espren','Hyloch','Aldwell']

In [ ]:
def ResExtraction(terr_data):
    res_data = terr_data['resources']
    res_types = ['EMERALD','ORE','CROP','WOOD','FISH']
    generation = []
    storage = []
    limit = []
    for res in res_types:
        specific_res_data = next((r for r in res_data if r.get('type') == res), None)
        if specific_res_data:
            generation.append(specific_res_data['generation'])
            storage.append(specific_res_data['storage'])
            limit.append(specific_res_data['limit'])
        else:
            generation.append(0)
            storage.append(0)
    return generation,storage,limit

In [ ]:
calculated_times = []

for city in list_of_cities:
    generation, storage, limit = ResExtraction(data[city])
    if limit[1] == 1200:
        usage_per_hour = 800
    elif limit[1] == 2400:
        usage_per_hour = 3000
    else:
        continue
    generation_per_hour = generation[0]
    current_ems_storage = storage[0]
    time_estimation = (current_ems_storage - generation_per_hour/60) / (generation_per_hour/3600 - usage_per_hour/3600)
    calculated_times.append(time_estimation)

median_time = round(statistics.mode(calculated_times))
for calculated_time in calculated_times:
    len(calculated_time)
    counter = 0
    if median_time - 0.4 <= calculated_time <= median_time + 0.4:
        counter += 1
    if counter >= len(calculated_times) * 0.6:
        final_time = median_time
        break

if counter < len(calculated_times) * 0.6: # fallback to median if mode is not representative
    median_time = statistics.median(calculated_times)
    final_time = median_time

print(f"Estimated time until next resource tick: {final_time} hours")